# Testes exploratórios – coletor Roblox e GOG
Cada célula de código abaixo é um teste independente. Todas usam o `EthicalHTTPClient` (User-Agent, rate limit de 1,5 s, retries).

## Teste 0 – Setup e imports

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "test":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from collectors.utils import EthicalHTTPClient, save_raw_json
from config.settings import USER_AGENT, RATE_LIMIT_SECONDS

client = EthicalHTTPClient()
print("Raiz do projeto:", ROOT)
print("User-Agent:", USER_AGENT)
print("Rate limit (s):", RATE_LIMIT_SECONDS)

Raiz do projeto: C:\Users\Silvan\Documents\Trabalho-Coleta-e-Raspagem\projeto-coleta-e-raspagem-de-dados
User-Agent: IMD1152-GameGenres-UFRN/1.0 (Academic Research - IMD/UFRN; https://github.com/VitorCostaID/projeto-coleta-e-raspagem-de-dados; vitor.costa.717@ufrn.edu.br)
Rate limit (s): 1.5


## Teste 1 – robots.txt dos hosts (ética)

In [2]:
for host in ["https://www.roblox.com", "https://www.gog.com", "https://catalog.gog.com", "https://embed.gog.com"]:
    try:
        txt = client.get_text(f"{host}/robots.txt")
        print(f"=== {host}/robots.txt ({len(txt)} chars) ===")
        print(txt[:800], "\n")
    except Exception as exc:
        print(f"=== {host}: falhou -> {exc}\n")

=== https://www.roblox.com/robots.txt (2281 chars) ===
user-agent: *

disallow: *.ashx
disallow: /abusereport/
disallow: /*/abusereport/
disallow: /admi/
disallow: /ads/
disallow: /catalog/contents
disallow: /catalog/html
disallow: /client-status
disallow: /data/
disallow: /error/
disallow: /forum/
disallow: /Forum/
disallow: /game/report-event
disallow: /game/report-stats
disallow: /ide/clienttoolbox
disallow: /javascript/
disallow: /reports/
disallow: /thumbnail/resolve-hash
disallow: /thumbnail_holder/g
disallow: /viewapp/
disallow: /error.html
disallow: /offline.html
disallow: /badges/
disallow: /*/badges/
disallow: /*.aspx$
disallow: /upgrades/checkout/success?giftPurchaseId=*
disallow: /report-abuse
disallow: /*/report-abuse

Sitemap: https://www.roblox.com/sitemap.xml
Sitemap: https://www.roblox.com/sitemap-games.xml
Sitemap: https://ww 

=== https://www.gog.com/robots.txt (152 chars) ===
User-agent: *


Disallow: /upload/
Disallow: /www/
Disallow: /downloader2/
Disallow: /galax

## Teste 2 – Roblox: listagem de jogos por sort

In [3]:
SID = "00000000-0000-0000-0000-000000000000"
BASE = "https://apis.roblox.com/explore-api/v1"
roblox_list = None
try:
    sorts = client.get_json(f"{BASE}/get-sorts", params={"sessionId": SID}).get("sorts", [])
    print("Sorts:", [s["sortId"] for s in sorts])
    roblox_list = client.get_json(f"{BASE}/get-sort-content", params={"sessionId": SID, "sortId": "top-playing-now"})
    games = roblox_list.get("games", [])
    print("Chaves:", list(roblox_list.keys()), "| jogos:", len(games))
    if games:
        print(json.dumps(games[0], indent=2, ensure_ascii=False)[:600])
except Exception as exc:
    print("Falhou:", exc)

Sorts: ['filters_v5', 'top-trending', 'up-and-coming', 'top-playing-now', 'fun-with-friends', 'top-revisited']
Chaves: ['contentType', 'gameSetTypeId', 'gameSetTargetId', 'primarySortId', 'secondarySortId', 'sortId', 'sortDisplayName', 'subtitle', 'topicLayoutData', 'treatmentType', 'games'] | jogos: 97
{
  "universeId": 10563114921,
  "rootPlaceId": 107778070777162,
  "name": "Roube um ovo",
  "playerCount": 420569,
  "totalUpVotes": 4329714,
  "totalDownVotes": 415279,
  "isSponsored": false,
  "nativeAdData": "",
  "minimumAge": 0,
  "ageRecommendationDisplayName": "Maturidade: Mínima",
  "contentMaturity": "minimal",
  "genreL1": "Simulation"
}


## Teste 3 – Roblox: detalhes por `universeIds` (playing, visits, genre)

In [4]:
universe_ids = [g["universeId"] for g in (roblox_list or {}).get("games", [])][:5]
if not universe_ids:
    universe_ids = [383310974]  # fallback: Adopt Me!
    print("Usando universeId de fallback:", universe_ids)
try:
    details = client.get_json("https://games.roblox.com/v1/games", params={"universeIds": ",".join(map(str, universe_ids))})
    rows = details.get("data", [])
    print("Registros:", len(rows))
    for r in rows:
        print({k: r.get(k) for k in ("id", "name", "playing", "visits", "genre", "genre_l1", "genre_l2", "favoritedCount")})
    print("Campos disponiveis:", sorted(rows[0].keys()) if rows else None)
except Exception as exc:
    print("Falhou:", exc)

Registros: 5
{'id': 10563114921, 'name': 'Roube um ovo', 'playing': 443054, 'visits': 11943661712, 'genre': 'All', 'genre_l1': 'Simulation', 'genre_l2': 'Tycoon', 'favoritedCount': 10350668}
{'id': 1686885941, 'name': 'Brookhaven 🏡 RP', 'playing': 329560, 'visits': 89809625949, 'genre': 'Town and City', 'genre_l1': 'Roleplay & Avatar Sim', 'genre_l2': 'Life', 'favoritedCount': 31176289}
{'id': 994732206, 'name': 'Blox Fruits', 'playing': 217508, 'visits': 65662439970, 'genre': 'Adventure', 'genre_l1': 'RPG', 'genre_l2': 'Action RPG', 'favoritedCount': 20296703}
{'id': 383310974, 'name': 'Adopt Me!', 'playing': 191826, 'visits': 45507934989, 'genre': 'RPG', 'genre_l1': 'Roleplay & Avatar Sim', 'genre_l2': 'Pet Care', 'favoritedCount': 29904666}
{'id': 7326934954, 'name': '99 Noites na Floresta 🔦', 'playing': 149402, 'visits': 30672025783, 'genre': 'All', 'genre_l1': 'Survival', 'genre_l2': '', 'favoritedCount': 10381328}
Campos disponiveis: ['allowedGearCategories', 'allowedGearGenres',

## Teste 4 – Roblox: limite de lote (até 100 IDs por chamada)

In [5]:
ids = [g["universeId"] for g in (roblox_list or {}).get("games", [])]
print(f"IDs disponiveis do teste 2: {len(ids)}")
for size in (100, 50, 25, 10):
    batch = ids[:size]
    if len(batch) < size:
        continue
    try:
        d = client.get_json("https://games.roblox.com/v1/games", params={"universeIds": ",".join(map(str, batch))})
        print(f"lote={size}: OK, recebidos {len(d.get('data', []))}")
        break
    except Exception as exc:
        print(f"lote={size}: FALHOU ({str(exc)[:60]})")

IDs disponiveis do teste 2: 97
lote=50: OK, recebidos 50


## Teste 5 – GOG: catálogo (catalog.gog.com)

In [6]:
GOG_CATALOG_URL = "https://catalog.gog.com/v1/catalog"
gog_catalog = None
try:
    gog_catalog = client.get_json(GOG_CATALOG_URL, params={
        "limit": 5, "page": 1, "order": "desc:trending",
        "productType": "in:game,pack",
    })
    print("Chaves de topo:", list(gog_catalog.keys()))
    print("Total de produtos:", gog_catalog.get("productCount"), "| páginas:", gog_catalog.get("pages"))
    prods = gog_catalog.get("products", [])
    if prods:
        print(json.dumps(prods[0], indent=2, ensure_ascii=False)[:1500])
except Exception as exc:
    print("Falhou:", exc)

Chaves de topo: ['pages', 'currentlyShownProductCount', 'productCount', 'products', 'dreamlistGames', 'filters', 'searchAlgo']
Total de produtos: 8486 | páginas: 1698
{
  "id": "1963606710",
  "slug": "lara_croft_and_the_guardian_of_light",
  "features": [
    {
      "name": "Achievements",
      "slug": "achievements"
    },
    {
      "name": "Controller support",
      "slug": "controller_support"
    },
    {
      "name": "Co-op",
      "slug": "coop"
    },
    {
      "name": "Leaderboards",
      "slug": "leaderboards"
    },
    {
      "name": "Multi-player",
      "slug": "multi"
    },
    {
      "name": "Overlay",
      "slug": "overlay"
    },
    {
      "name": "Single-player",
      "slug": "single"
    }
  ],
  "screenshots": [
    "https://images.gog-statics.com/e2b3f2e4739c9d4451c36fe2c91ac394f9e96b55d94aa11600656cda9852b67e_{formatter}.jpg",
    "https://images.gog-statics.com/867b6a090662f47207b0822b6cc31b4831cc432fe67bfe5db1eee91dbe598f01_{formatter}.jpg",
   

## Teste 6 – GOG: endpoint alternativo (embed.gog.com)

In [7]:
try:
    alt = client.get_json("https://embed.gog.com/games/ajax/filtered", params={"mediaType": "game", "page": 1, "sort": "popularity"})
    print("Chaves de topo:", list(alt.keys()))
    print("totalGamesFound:", alt.get("totalGamesFound"), "| totalPages:", alt.get("totalPages"))
    prods = alt.get("products", [])
    if prods:
        print(json.dumps(prods[0], indent=2, ensure_ascii=False)[:1200])
except Exception as exc:
    print("Falhou:", exc)

Chaves de topo: ['products', 'ts', 'page', 'totalPages', 'totalResults', 'totalGamesFound', 'totalMoviesFound']
totalGamesFound: 20274 | totalPages: 423
{
  "id": 1841631965,
  "releaseDate": 1551304800,
  "availability": {
    "isAvailable": true,
    "isAvailableInAccount": true
  },
  "salesVisibility": {
    "isActive": true,
    "fromObject": {
      "date": "2024-07-12 14:55:00.000000",
      "timezone_type": 1,
      "timezone": "+02:00"
    },
    "from": 1720788900,
    "toObject": null,
    "to": null
  },
  "buyable": true,
  "title": " Offworld Trading Company - Market Corrections",
  "image": "//images-3.gog-statics.com/51ea7e3c7030f97e04e9a9b37afcb1c53a60537f2f496d3982fad21b44121b5d",
  "url": "/en/game/offworld_trading_company_market_corrections",
  "supportUrl": "/support/offworld_trading_company_market_corrections",
  "forumUrl": "/forum/offworld_trading_company",
  "worksOn": {
    "Windows": true,
    "Mac": true,
    "Linux": false
  },
  "category": "Simulation",
 

## Teste 7 – GOG: paginação (página 1 vs 2 não se repetem)

In [8]:
def gog_page(p):
    return client.get_json(GOG_CATALOG_URL, params={"limit": 5, "page": p, "order": "desc:trending",
                           "productType": "in:game,pack"})
try:
    p1, p2 = gog_page(1), gog_page(2)
    ids1 = {p["id"] for p in p1.get("products", [])}
    ids2 = {p["id"] for p in p2.get("products", [])}
    print("Página 1:", ids1); print("Página 2:", ids2)
    print("Interseção (esperado vazio):", ids1 & ids2)
except Exception as exc:
    print("Falhou:", exc)

Página 1: {'1963606710', '2060118552', '1207658787', '1680301192', '1274966284'}
Página 2: {'1440163901', '1993442013', '1207659026', '1901861030', '2093619782'}
Interseção (esperado vazio): set()


## Teste 8 – Persistência: `save_raw_json` em diretório temporário

In [9]:
import tempfile
sample = [{"id": 1, "name": "Teste"}, {"id": 2, "name": "Outro"}]
with tempfile.TemporaryDirectory() as tmp:
    out = save_raw_json(sample, "sample_raw.json", platform="teste", output_dir=Path(tmp))
    data = json.loads(out.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False))
    assert all("platform" in r and "collected_at" in r for r in data)
    print("OK: metadados platform e collected_at presentes.")

[
  {
    "id": 1,
    "name": "Teste",
    "platform": "teste",
    "collected_at": "2026-11-25T23:51:36.896722+00:00"
  },
  {
    "id": 2,
    "name": "Outro",
    "platform": "teste",
    "collected_at": "2026-11-25T23:51:36.896722+00:00"
  }
]
OK: metadados platform e collected_at presentes.


## Teste 9 – Rate limit: intervalo entre requisições

In [10]:
import time
t0 = time.monotonic()
for _ in range(3):
    client.get_text("https://www.gog.com/robots.txt")
elapsed = time.monotonic() - t0
print(f"3 requisições em {elapsed:.2f}s (mínimo esperado ≈ {2 * RATE_LIMIT_SECONDS:.2f}s)")
assert elapsed >= 2 * RATE_LIMIT_SECONDS * 0.95, "Rate limit não está sendo respeitado"
print("OK")

3 requisições em 3.09s (mínimo esperado ≈ 3.00s)
OK
